# Belief-strategy consistency: evidence of manipulation

`belief_analysis.ipynb` established that in the treatment arm, a participant's `collabBelief` is shown to their partner via the robot's info modal (`"Partner Belief = " + partnerCollabBelief`), and that the *partner's* belief predicts a participant's own strategy choice specifically in treatment (the `partner_belief_c:arm` interaction). That creates an incentive that only exists in treatment: a participant could report an inflated `collabBelief` -- signalling they'll collaborate, to induce their partner into choosing `C` -- while actually planning to choose `I` themselves.

This is a viable exploit because of the payoff asymmetry documented throughout this analysis: `V_Y` (the individual design's payoff) doesn't depend on the partner's choice at all, so reporting a high belief and then defecting is never worse for the person doing it, and is only ever costly to their partner. This notebook asks: is there evidence of individual participants doing this systematically, more than the group as a whole?

**Operationalizing "inconsistent."** Rather than an arbitrary threshold, this reuses the risk-dominance measure `u` from `risk_dominance_analysis.ipynb` -- for a given task, `u` is the minimum belief in the partner's cooperation needed to make choosing collaboratively the rational (higher expected-value) choice, given only *that participant's own* task payoffs:

$$u = \frac{V_Y^{II} - V_A^{CI}}{(V_Y^{II} - V_A^{CI}) + (V_A^{CC} - V_Y^{IC})}$$

For each (participant, round), `belief_favors_C` = `collabBelief >= 100 * u_own`: the participant's own stated belief was high enough that, by their own payoff math, collaborating was the better expected bet. `inconsistent_exploit` = `belief_favors_C` and the participant nonetheless chose `I`. Only `u_own` (the participant's own task) is used, not the round's paired `R` from the risk-dominance notebook -- `R` mixes in the *partner's* task difficulty, which the participant reporting the belief has no way to act strategically on for their own payoff calculation.

This is restricted to the **treatment arm only**: control has no belief-sharing channel, so there's no mechanism (or incentive) for this pattern to mean the same thing there.

**What this can and can't show.** A single inconsistent round doesn't prove intent -- it could be poor calibration, a change of mind, or noise. But the directional asymmetry (only *this* direction benefits the reporter at their partner's expense) plus testing each participant's *rate* of the behavior against the group as a whole is a reasonable operationalization of "evidence consistent with manipulation," not proof of it for any individual round.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

task = pd.read_csv("task_data.csv")
task_summary = pd.read_csv("task_summary.csv")

# task_summary.csv writes "n/a" for distraction/training tasks, which read_csv
# loads as NaN by default -- filter with notna(), not a string comparison
# (see risk_dominance_analysis.ipynb).
task_summary = task_summary[task_summary["task_difficulty"].notna()].copy()
task_summary["u"] = (task_summary["V_Y_II"] - task_summary["V_A_CI"]) / (
    (task_summary["V_Y_II"] - task_summary["V_A_CI"]) + (task_summary["V_A_CC"] - task_summary["V_Y_IC"])
)
u_by_index = task_summary.set_index("task_index")["u"]

## Drop missing data, restrict to treatment, reshape to one row per (participant, round)

Same drop criterion as the other outcome notebooks. Then restricted to `arm == "treatment"` for the reasons above, and reshaped exactly as in `belief_analysis.ipynb` -- one row per participant per round, since belief and strategy are individual, not joint, measures.

In [2]:
missing = (task["strategy_1"] == "undefined") | (task["strategy_2"] == "undefined")
task = task[~missing].copy()
task["pair_id"] = task["username_1"] + "_" + task["username_2"]

treatment = task[task["arm"] == "treatment"].copy()

shared_cols = ["pair_id", "round"]
belief = pd.concat([
    treatment[shared_cols + ["username_1", "collabBelief_1", "strategy_1", "task_1"]].rename(
        columns={"username_1": "username", "collabBelief_1": "collabBelief",
                 "strategy_1": "strategy", "task_1": "task"}),
    treatment[shared_cols + ["username_2", "collabBelief_2", "strategy_2", "task_2"]].rename(
        columns={"username_2": "username", "collabBelief_2": "collabBelief",
                 "strategy_2": "strategy", "task_2": "task"}),
], ignore_index=True)

belief["u_own"] = belief["task"].map(u_by_index)
belief["belief_favors_C"] = belief["collabBelief"] >= belief["u_own"] * 100
belief["inconsistent_exploit"] = belief["belief_favors_C"] & (belief["strategy"] == "I")
belief["inconsistent_generous"] = (~belief["belief_favors_C"]) & (belief["strategy"] == "C")

print(f"n={len(belief)} (participant, round) observations, {belief['username'].nunique()} participants, "
      f"{belief['pair_id'].nunique()} pairs, all treatment")

n=840 (participant, round) observations, 28 participants, 14 pairs, all treatment


## Base rates

`inconsistent_generous` (the reverse pattern -- belief implied `I` was rational, but the participant chose `C` anyway) is included only for context, since the payoff asymmetry means it can't be an exploit of the partner in the same way; the rest of this notebook focuses on `inconsistent_exploit`.

In [3]:
n_favorable = belief["belief_favors_C"].sum()
n_exploit = belief["inconsistent_exploit"].sum()
n_generous = belief["inconsistent_generous"].sum()

print(f"belief_favors_C:        {n_favorable} / {len(belief)} rounds ({n_favorable / len(belief):.1%})")
print(f"inconsistent_exploit:   {n_exploit} / {n_favorable} belief-favors-C rounds "
      f"({n_exploit / n_favorable:.1%} of those, {n_exploit / len(belief):.1%} of all rounds)")
print(f"inconsistent_generous:  {n_generous} / {len(belief) - n_favorable} belief-favors-I rounds "
      f"({n_generous / (len(belief) - n_favorable):.1%} of those)")

belief_favors_C:        680 / 840 rounds (81.0%)
inconsistent_exploit:   54 / 680 belief-favors-C rounds (7.9% of those, 6.4% of all rounds)
inconsistent_generous:  82 / 160 belief-favors-I rounds (51.2% of those)


## Flagging individual participants

For each participant, restricted to their own `belief_favors_C` rounds (the only rounds where `inconsistent_exploit` is even possible), compare their inconsistency rate against a **leave-one-out population baseline** (the rate across all *other* participants' `belief_favors_C` rounds) with a one-sided exact binomial test -- excluding each participant from their own baseline avoids a participant's own extreme behavior making the test conservative for them specifically. With 28 participants tested, p-values are FDR-corrected (Benjamini-Hochberg) rather than read individually.

This treats each participant's own rounds as independent Bernoulli trials, which is a simplification -- a participant's rounds may be serially correlated (e.g. consistently exploitative once they start) rather than i.i.d. Read the test as a descriptive flagging tool for "how extreme is this rate relative to the group," not a fully specified hypothesis test.

In [4]:
favorable = belief[belief["belief_favors_C"]]

per_user = favorable.groupby("username").agg(
    n_favorable=("inconsistent_exploit", "size"),
    n_inconsistent=("inconsistent_exploit", "sum"),
)
per_user = per_user[per_user["n_favorable"] > 0].copy()
per_user["rate"] = per_user["n_inconsistent"] / per_user["n_favorable"]

total_favorable = len(favorable)
total_inconsistent = favorable["inconsistent_exploit"].sum()

p_values = []
for _, row in per_user.iterrows():
    loo_rate = (total_inconsistent - row["n_inconsistent"]) / (total_favorable - row["n_favorable"])
    result = stats.binomtest(int(row["n_inconsistent"]), int(row["n_favorable"]), loo_rate, alternative="greater")
    p_values.append(result.pvalue)
per_user["p_value"] = p_values

_, per_user["p_adj_fdr"], _, _ = multipletests(per_user["p_value"], method="fdr_bh")
per_user["flagged"] = per_user["p_adj_fdr"] < 0.05

per_user.sort_values("p_value").round(4)

,n_favorable,n_inconsistent,rate,p_value,p_adj_fdr,flagged
username,,,,,,
user0031,30,14,0.4667,0.0000,0.0000,True
user0042,25,10,0.4000,0.0000,0.0000,True
user0048,16,8,0.5000,0.0000,0.0000,True
user0049,12,4,0.3333,0.0095,0.0667,False
user0050,13,4,0.3077,0.0130,0.0729,False
user0041,17,3,0.1765,0.1380,0.6438,False
user0061,30,4,0.1333,0.1963,0.7304,False
user0032,11,2,0.1818,0.2087,0.7304,False
user0047,26,3,0.1154,0.3309,1.0000,False


## Does a participant's inconsistency rate relate to their pair's overall success?

A participant flagged here is, by construction, someone who often chose `I` when their own stated belief said `C` was the better bet -- so their pair should tend to under-perform relative to what mutual collaboration would have achieved. This checks that directly: each participant's `inconsistent_exploit` rate against their pair's overall successful-collaboration rate (`C`/`C` rounds, out of all 30 real-task rounds for that pair).

In [5]:
treatment["success"] = ((treatment["strategy_1"] == "C") & (treatment["strategy_2"] == "C")).astype(int)
pair_success = treatment.groupby("pair_id")["success"].mean()

username_to_pair = pd.concat([
    treatment[["username_1", "pair_id"]].rename(columns={"username_1": "username"}),
    treatment[["username_2", "pair_id"]].rename(columns={"username_2": "username"}),
]).drop_duplicates().set_index("username")["pair_id"]

per_user["pair_id"] = per_user.index.map(username_to_pair)
per_user["pair_success_rate"] = per_user["pair_id"].map(pair_success)

r, p = stats.pearsonr(per_user["rate"], per_user["pair_success_rate"])
print(f"Pearson r = {r:.3f}, p = {p:.4f} (n={len(per_user)} participants)")

Pearson r = -0.888, p = 0.0000 (n=28 participants)


For context, per-pair robot-use rate alongside success rate (referenced in the interpretation below, to check whether the flagged participants' pairs also stand out on robot use, as found independently in `robot_use_analysis.ipynb`).

In [6]:
treatment["either_used_robot"] = (treatment["usedRobot_1"] | treatment["usedRobot_2"]).astype(int)
per_pair_context = treatment.groupby("pair_id").agg(
    robot_use_rate=("either_used_robot", "mean"), success_rate=("success", "mean"),
)
per_pair_context.sort_values("robot_use_rate", ascending=False).round(3)

,robot_use_rate,success_rate
pair_id,,
user0017_user0018,1.000,1.000
user0041_user0042,1.000,0.400
user0047_user0048,0.733,0.333
user0061_user0062,0.633,0.800
user0033_user0034,0.533,1.000
user0049_user0050,0.467,0.300
user0015_user0016,0.233,1.000
user0055_user0056,0.200,1.000
user0057_user0058,0.167,1.000


## Interpretation

**Base rates.** 81.0% of treatment (participant, round) observations had `belief_favors_C` -- most stated beliefs were high enough to make collaborating the rational choice by the participant's own payoff math. Among those, 7.9% (54 of 680) were followed by choosing `I` anyway (`inconsistent_exploit`) -- the pattern potentially consistent with exploiting the belief-sharing mechanism. For context, the reverse pattern (`inconsistent_generous`) is actually far more common relative to its opportunity: 51.3% (82 of 160) of belief-favors-`I` rounds still ended in `C` -- participants defaulted toward collaborating more often than their own stated belief implied was rational, the opposite of a manipulative pattern.

**Individual flagging.** Of 28 treatment participants with at least one `belief_favors_C` round, three survive FDR correction as significantly more exploitative than the leave-one-out population baseline: `user0031` (14 of 30 favorable rounds, 46.7%), `user0042` (10 of 25, 40.0%), and `user0048` (8 of 16, 50.0%). Two more sit at an elevated but not-significant rate after correction -- `user0049` (4/12, 33.3%, p_adj ≈ 0.067) and `user0050` (4/13, 30.8%, p_adj ≈ 0.073) -- worth noting as a second tier rather than treating as confirmed. The remaining 18 of 28 participants show *zero* inconsistent rounds at all; the behavior is concentrated in a small minority, not spread across the sample.

**Corroboration.** A participant's `inconsistent_exploit` rate correlates strongly with their pair's overall successful-collaboration rate (Pearson r = -0.888, p < 0.001, n = 28) -- pairs with more exploitative behavior collaborate less overall. Part of this is close to definitional (an `I` choice can't itself be part of a `C`/`C` round, whether or not it was "inconsistent"), so this alone isn't independent proof of manipulation. What *is* independent, and more telling: the three flagged participants line up by name with pairs already flagged as anomalous elsewhere in this analysis, using entirely different data. `user0031` is from the `user0031_user0032` pair singled out in `outcome_analysis.ipynb`'s robustness check for behaving like a control pair despite being assigned to treatment, and confirmed here as one of only two treatment pairs with a 0% robot-use rate. `user0042` and `user0048` are from `user0041_user0042` and `user0047_user0048` -- the pairs with the highest (100%) and third-highest (73.3%) robot-use rates in the whole treatment arm, and two of its three lowest success rates (40.0% and 33.3%, against a treatment median of 100%). Three separate analyses, using separate variables (outcome rates, robot-use frequency, belief-strategy consistency), converge on the same small set of pairs -- that convergence is the strongest evidence here, stronger than any single measure alone.

**Caveats.** This flags *patterns* consistent with manipulation, not intent -- `u_own` assumes a stylized expected-value-maximizing model of what's "rational," and a participant may not reason about their own payoffs this precisely. The binomial test also treats each participant's rounds as independent trials, which likely understates the true variability of a serially-correlated behavior. And per-participant power varies a great deal (7 to 30 favorable rounds) -- a participant who never gets flagged might simply have had few opportunities to be. Given all that, this is best read as a data-driven shortlist for closer qualitative review (e.g. by round-by-round timeline) rather than a final determination of who manipulated whom.